# LAB 10 — Tokenization, Offset Mapping & Answer Span Alignment

Notebook này triển khai các phần kỹ thuật cốt lõi của LAB 10 (Mục 10, 11, 12, 13, 15, 16):
1. Tokenization cặp `(question, context)` và kiểm tra `input_ids`, `attention_mask`, `token_type_ids`.
2. Phân biệt `character index`, `word index`, `token index` và dùng `offset_mapping`.
3. Xử lý context dài bằng **sliding window** (`truncation="only_second"`, `stride`, `return_overflowing_tokens=True`).
4. Ánh xạ **Answer Span Alignment** bằng hàm `align_answer_to_tokens` tự cài đặt trong `qa_model.py` và kiểm thử đầy đủ **4 edge cases**:
   - Answer bắt đầu giữa token;
   - Answer kết thúc giữa token;
   - Answer bị cắt ngang (truncation);
   - Answer nằm hoàn toàn ngoài window.


In [1]:
import inspect
from datasets import load_dataset
from transformers import AutoTokenizer

from qa_model import align_answer_to_tokens

# Load DistilBERT (backbone chính dùng cho LAB 10) và BERT-base (để so sánh token_type_ids)
distil_tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print("DistilBERT fast tokenizer:", distil_tok.is_fast)
print("BERT fast tokenizer      :", bert_tok.is_fast)


/Users/user/.cache/nlp_lab10_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DistilBERT fast tokenizer: True
BERT fast tokenizer      : True


## 1. Tokenization của Extractive QA & Lưu ý về `token_type_ids` (Mục 10)

Input của bài toán QA ghép câu hỏi và đoạn văn bản theo dạng:
```text
[CLS] question tokens [SEP] context tokens [SEP]
```
Một điểm rất dễ nhầm là mặc định nghĩ Transformer nào cũng trả về `token_type_ids`. Thực tế `bert-base-uncased` có `token_type_ids` (0 cho phần question, 1 cho phần context), còn `distilbert-base-uncased` (và RoBERTa) **không sử dụng** `token_type_ids` — thay vào đó ta dùng `encoding.sequence_ids()` để biết token nào thuộc question (`0`), token nào thuộc context (`1`), và token nào là special token (`None`).


In [2]:
question = "What is the capital of France?"
context = "The capital of France is Paris."

bert_enc = bert_tok(question, context)
distil_enc = distil_tok(question, context)

print("BERT keys      :", list(bert_enc.keys()))
print("DistilBERT keys:", list(distil_enc.keys()))

print("\n--- BERT tokens & token_type_ids ---")
bert_tokens = bert_tok.convert_ids_to_tokens(bert_enc["input_ids"])
for i, (tok, tid, mask, ttype) in enumerate(
    zip(bert_tokens, bert_enc["input_ids"], bert_enc["attention_mask"], bert_enc["token_type_ids"])
):
    print(f"idx={i:2d} | token={tok:10s} | id={tid:5d} | mask={mask} | token_type_id={ttype}")

print("\n--- DistilBERT tokens & sequence_ids() ---")
distil_tokens = distil_tok.convert_ids_to_tokens(distil_enc["input_ids"])
for i, (tok, tid, mask, sid) in enumerate(
    zip(distil_tokens, distil_enc["input_ids"], distil_enc["attention_mask"], distil_enc.sequence_ids())
):
    print(f"idx={i:2d} | token={tok:10s} | id={tid:5d} | mask={mask} | sequence_id={sid}")


BERT keys      : ['input_ids', 'token_type_ids', 'attention_mask']
DistilBERT keys: ['input_ids', 'attention_mask']

--- BERT tokens & token_type_ids ---
idx= 0 | token=[CLS]      | id=  101 | mask=1 | token_type_id=0
idx= 1 | token=what       | id= 2054 | mask=1 | token_type_id=0
idx= 2 | token=is         | id= 2003 | mask=1 | token_type_id=0
idx= 3 | token=the        | id= 1996 | mask=1 | token_type_id=0
idx= 4 | token=capital    | id= 3007 | mask=1 | token_type_id=0
idx= 5 | token=of         | id= 1997 | mask=1 | token_type_id=0
idx= 6 | token=france     | id= 2605 | mask=1 | token_type_id=0
idx= 7 | token=?          | id= 1029 | mask=1 | token_type_id=0
idx= 8 | token=[SEP]      | id=  102 | mask=1 | token_type_id=0
idx= 9 | token=the        | id= 1996 | mask=1 | token_type_id=1
idx=10 | token=capital    | id= 3007 | mask=1 | token_type_id=1
idx=11 | token=of         | id= 1997 | mask=1 | token_type_id=1
idx=12 | token=france     | id= 2605 | mask=1 | token_type_id=1
idx=13 | token

## 2. Phân biệt Character Index, Word Index và Token Index (Mục 11 & 12)

Dataset SQuAD cho đáp án dưới dạng `answer_start` (vị trí ký tự) và `answer_text`, trong khi Transformer chỉ làm việc trên `token index`. Dưới đây là kiểm chứng trực tiếp trên câu `"The capital of France is Paris."` (không nhầm `character offset` với `token index`).


In [3]:
answer_text = "Paris"
char_start = context.index(answer_text)
char_end = char_start + len(answer_text)
words = context.split()

print(f"Context        : {context!r}")
print(f"Character span : [{char_start}, {char_end}) -> {context[char_start:char_end]!r}")
print(f"Word list      : {words}")
print(f"Word index     : {words.index('Paris.')} (lưu ý dính dấu chấm 'Paris.')")

enc_with_offsets = distil_tok(
    question,
    context,
    return_offsets_mapping=True,
)
tokens = distil_tok.convert_ids_to_tokens(enc_with_offsets["input_ids"])
offsets = enc_with_offsets["offset_mapping"]
seq_ids = enc_with_offsets.sequence_ids()

print("\nTokens thuộc context (sequence_id == 1) kèm offset_mapping:")
for idx, (tok, off, sid) in enumerate(zip(tokens, offsets, seq_ids)):
    if sid == 1:
        print(f"  token_idx={idx:2d} | token={tok:8s} | char_offset={off} -> {context[off[0]:off[1]]!r}")


Context        : 'The capital of France is Paris.'
Character span : [25, 30) -> 'Paris'
Word list      : ['The', 'capital', 'of', 'France', 'is', 'Paris.']
Word index     : 5 (lưu ý dính dấu chấm 'Paris.')

Tokens thuộc context (sequence_id == 1) kèm offset_mapping:
  token_idx= 9 | token=the      | char_offset=(0, 3) -> 'The'
  token_idx=10 | token=capital  | char_offset=(4, 11) -> 'capital'
  token_idx=11 | token=of       | char_offset=(12, 14) -> 'of'
  token_idx=12 | token=france   | char_offset=(15, 21) -> 'France'
  token_idx=13 | token=is       | char_offset=(22, 24) -> 'is'
  token_idx=14 | token=paris    | char_offset=(25, 30) -> 'Paris'
  token_idx=15 | token=.        | char_offset=(30, 31) -> '.'


## 3. Core Implementation: Sliding Window & Overflowing Tokens (Mục 13 & 15)

Khi $T_{\text{context}} > T_{\max}$, nếu chỉ cắt cứng (`truncation=True`) thì đáp án ở cuối đoạn văn sẽ bị mất hoàn toàn. Vì vậy ta dùng **sliding window** với:
- `truncation="only_second"`: chỉ cắt phần `context`, luôn giữ nguyên toàn bộ `question`.
- `max_length=384`, `stride=128`: mỗi window tối đa 384 token, hai window liên tiếp chồng lấn nhau 128 token của context để tránh đáp án bị cắt đứt ngay ranh giới window.
- `return_overflowing_tokens=True`: trả về mọi window thay vì vứt bỏ phần tràn.
- `return_offsets_mapping=True`: trả về cặp `(char_start, char_end)` cho từng token.


In [4]:
# Tạo một context dài hơn 384 tokens để quan sát nhiều sliding windows
long_context = (
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It is named after the engineer Gustave Eiffel, whose company designed and built the tower. "
) * 18 + (
    "In 1889, the tower was erected as the entrance arch to the 1889 World's Fair, "
    "and it initially received criticism from some of France's leading artists and intellectuals. "
    "Today, the tower is 330 metres tall and is the most visited monument with an entrance fee in the world."
)
long_question = "How tall is the Eiffel Tower today?"

inputs = distil_tok(
    long_question,
    long_context,
    truncation="only_second",
    max_length=384,
    stride=128,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

print("Keys trả về:", list(inputs.keys()))
print("Số windows sinh ra          :", len(inputs["input_ids"]))
print("overflow_to_sample_mapping  :", inputs["overflow_to_sample_mapping"])

for w_idx in range(len(inputs["input_ids"])):
    ids = inputs["input_ids"][w_idx]
    mask = inputs["attention_mask"][w_idx]
    offs = inputs["offset_mapping"][w_idx]
    sids = inputs.sequence_ids(w_idx)
    ctx_positions = [i for i, s in enumerate(sids) if s == 1]
    first_char = offs[ctx_positions[0]][0]
    last_char = offs[ctx_positions[-1]][1]
    print(
        f"Window {w_idx}: total_tokens={len(ids)}, active_mask={sum(mask)}, "
        f"context_tokens={len(ctx_positions)}, char_range=[{first_char}, {last_char})"
    )

if len(inputs["input_ids"]) >= 2:
    sids0 = inputs.sequence_ids(0)
    sids1 = inputs.sequence_ids(1)
    ctx0_end = inputs["offset_mapping"][0][[i for i, s in enumerate(sids0) if s == 1][-1]][1]
    ctx1_start = inputs["offset_mapping"][1][[i for i, s in enumerate(sids1) if s == 1][0]][0]
    print(f"\nOverlap ký tự giữa Window 0 và Window 1: [{ctx1_start}, {ctx0_end}) -> dài {ctx0_end - ctx1_start} ký tự")


Keys trả về: ['input_ids', 'attention_mask', 'offset_mapping', 'overflow_to_sample_mapping']
Số windows sinh ra          : 3
overflow_to_sample_mapping  : [0, 0, 0]
Window 0: total_tokens=384, active_mask=384, context_tokens=371, char_range=[0, 1616)
Window 1: total_tokens=384, active_mask=384, context_tokens=371, char_range=[1063, 2683)
Window 2: total_tokens=322, active_mask=322, context_tokens=309, char_range=[2118, 3496)

Overlap ký tự giữa Window 0 và Window 1: [1063, 1616) -> dài 553 ký tự


### Giải thích 4 trường quan trọng của `inputs`
- **`input_ids`**: Danh sách các window; mỗi window là chuỗi ID số nguyên biểu diễn `[CLS] question [SEP] context_window [SEP]`.
- **`attention_mask`**: Gồm `1` cho token thật và `0` cho token padding (khi bật `padding`). Giúp self-attention không chú ý vào vị trí đệm.
- **`offset_mapping`**: Với mỗi token ở vị trí `t`, lưu cặp `(start_char, end_char)` tương ứng trong chuỗi gốc (với special token `[CLS]`, `[SEP]` thì là `(0, 0)`). Nhờ bảng này ta mới đổi được character span sang token span và ngược lại.
- **`overflow_to_sample_mapping`**: Cho biết mỗi window thuộc về mẫu dữ liệu gốc số mấy trong batch (ví dụ `[0, 0]` nghĩa là cả 2 window đều cắt ra từ mẫu thứ `0`). Rất quan trọng khi đánh giá vì ta phải gom dự đoán từ mọi window của cùng một mẫu rồi mới chọn span có điểm cao nhất.


## 4. Answer Span Alignment & Kiểm thử 4 Edge Cases (Mục 16)

Hàm `align_answer_to_tokens` được cài đặt tại `qa_model.py` (không dùng HF `pipeline`), xử lý ánh xạ `(answer_start, answer_end)` → `(start_token, end_token)` và phát hiện rõ cả 4 trường hợp biên theo đúng yêu cầu đề bài.


In [5]:
print(inspect.getsource(align_answer_to_tokens))


def align_answer_to_tokens(
    offsets: Sequence[Tuple[int, int]],
    sequence_ids: Sequence[Optional[int]],
    answer_start: int,
    answer_end: int,
    cls_index: int = 0,
) -> Dict[str, Any]:
    """Map character span [answer_start, answer_end) into token indices (start, end).

    Handles the 4 required cases:
    - Answer starts in the middle of a token.
    - Answer ends in the middle of a token.
    - Answer is truncated by window boundary (partially inside window).
    - Answer lies completely outside the current window.
    """
    if answer_end <= answer_start:
        return {
            "start_token": cls_index,
            "end_token": cls_index,
            "status": "empty_answer",
            "starts_mid_token": False,
            "ends_mid_token": False,
        }

    # Find contiguous context token bounds (sequence_ids == 1).
    context_start = 0
    while context_start < len(sequence_ids) and sequence_ids[context_start] != 1:
        context_start += 1

    i

In [6]:
def inspect_case(title: str, question: str, context: str, answer_text: str, answer_start: int,
                 max_length: int = 384, stride: int = 128, return_overflowing: bool = False):
    print("=" * 78)
    print(title)
    answer_end = answer_start + len(answer_text)
    print(f"Answer: {answer_text!r} at chars [{answer_start}, {answer_end})")
    enc = distil_tok(
        question,
        context,
        truncation="only_second",
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=return_overflowing,
        return_offsets_mapping=True,
    )

    num_windows = len(enc["input_ids"]) if return_overflowing else 1
    for w in range(num_windows):
        ids = enc["input_ids"][w] if return_overflowing else enc["input_ids"]
        offs = enc["offset_mapping"][w] if return_overflowing else enc["offset_mapping"]
        sids = enc.sequence_ids(w)
        res = align_answer_to_tokens(offs, sids, answer_start, answer_end, cls_index=0)
        toks = distil_tok.convert_ids_to_tokens(ids)
        span_toks = toks[res["start_token"] : res["end_token"] + 1]
        if res["status"] == "aligned":
            rec_start = offs[res["start_token"]][0]
            rec_end = offs[res["end_token"]][1]
            reconstructed = context[rec_start:rec_end]
        else:
            reconstructed = "<NOT IN THIS WINDOW -> CLS>"
        print(
            f"  Window {w}: status={res['status']}, "
            f"starts_mid={res['starts_mid_token']}, ends_mid={res['ends_mid_token']}, "
            f"token_span=({res['start_token']}, {res['end_token']}), "
            f"tokens={span_toks}, reconstructed={reconstructed!r}"
        )


# --- Kiểm thử chuẩn + 4 Edge Cases theo Mục 16 ---

# 0. Trường hợp chuẩn
c0 = "The capital of France is Paris."
inspect_case(
    "CASE 0 (Chuẩn): Exact token boundary",
    "What is the capital of France?", c0, "Paris", c0.index("Paris")
)

# 1. Edge Case 1: Answer bắt đầu giữa token
c1 = "The experiment yielded an unbelievable result in 2026."
inspect_case(
    "EDGE CASE 1: Answer bắt đầu giữa token ('believable' bên trong 'unbelievable')",
    "What kind of result was it?", c1, "believable", c1.index("believable")
)

# 2. Edge Case 2: Answer kết thúc giữa token ('1990' bên trong '1990s')
c2 = "The electronic music festival grew rapidly during the 1990s in Europe."
inspect_case(
    "EDGE CASE 2: Answer kết thúc giữa token ('1990' bên trong '1990s')",
    "When did the festival grow rapidly?", c2, "1990", c2.index("1990")
)

# 3. Edge Case 3: Answer bị cắt ngang do truncation (max_length=40 cắt ngay giữa cụm đáp án)
c3 = (
    "In the northern region of the country, "
    + ("many historical castles were built along the river. " * 2)
    + "The famous capital city is Reykjavik."
)
inspect_case(
    "EDGE CASE 3: Answer bị cắt ngang bởi truncation (max_length=40)",
    "What is the famous capital city?", c3, "famous capital city is Reykjavik",
    c3.index("famous capital city is Reykjavik"), max_length=40, stride=0, return_overflowing=False
)

# 4. Edge Case 4: Answer nằm hoàn toàn ngoài Window 0 (và được bắt lại ở Window sau nhờ stride)
c4 = (
    "This is filler background information about geography and history. " * 12
) + "The actual secret answer is Mount Fuji."
inspect_case(
    "EDGE CASE 4: Answer nằm ngoài Window 0, bắt được ở Window sau nhờ sliding window",
    "What is the actual secret answer?", c4, "Mount Fuji",
    c4.index("Mount Fuji"), max_length=64, stride=24, return_overflowing=True
)


CASE 0 (Chuẩn): Exact token boundary
Answer: 'Paris' at chars [25, 30)
  Window 0: status=aligned, starts_mid=False, ends_mid=False, token_span=(14, 14), tokens=['paris'], reconstructed='Paris'
EDGE CASE 1: Answer bắt đầu giữa token ('believable' bên trong 'unbelievable')
Answer: 'believable' at chars [28, 38)
  Window 0: status=aligned, starts_mid=True, ends_mid=False, token_span=(13, 13), tokens=['unbelievable'], reconstructed='unbelievable'
EDGE CASE 2: Answer kết thúc giữa token ('1990' bên trong '1990s')
Answer: '1990' at chars [54, 58)
  Window 0: status=aligned, starts_mid=False, ends_mid=True, token_span=(17, 17), tokens=['1990s'], reconstructed='1990s'
EDGE CASE 3: Answer bị cắt ngang bởi truncation (max_length=40)
Answer: 'famous capital city is Reykjavik' at chars [147, 179)
  Window 0: status=truncated, starts_mid=False, ends_mid=False, token_span=(0, 0), tokens=['[CLS]'], reconstructed='<NOT IN THIS WINDOW -> CLS>'
EDGE CASE 4: Answer nằm ngoài Window 0, bắt được ở Window 

## 5. Kiểm chứng trên dữ liệu SQuAD thật

Kiểm tra nhanh `align_answer_to_tokens` trên 500 mẫu đầu tiên của tập `SQuAD` train với hai cấu hình `(384, 128)` và `(256, 64)` để quan sát tỷ lệ phủ mẫu và số ca subtoken boundary trên dữ liệu thực tế.


In [7]:
squad_sample = load_dataset("rajpurkar/squad", split="train[:500]")

for ml, st in [(384, 128), (256, 64)]:
    exact_count = 0
    mid_token_count = 0
    truncated_count = 0
    out_count = 0
    total_windows = 0
    samples_covered = 0

    for ex in squad_sample:
        q = ex["question"]
        c = ex["context"]
        ans_text = ex["answers"]["text"][0]
        ans_start = ex["answers"]["answer_start"][0]
        ans_end = ans_start + len(ans_text)

        enc = distil_tok(
            q,
            c,
            truncation="only_second",
            max_length=ml,
            stride=st,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
        )

        found_for_sample = False
        for w in range(len(enc["input_ids"])):
            total_windows += 1
            res = align_answer_to_tokens(
                enc["offset_mapping"][w], enc.sequence_ids(w), ans_start, ans_end, cls_index=0
            )
            if res["status"] == "aligned":
                found_for_sample = True
                if res["starts_mid_token"] or res["ends_mid_token"]:
                    mid_token_count += 1
                else:
                    exact_count += 1
            elif res["status"] == "truncated":
                truncated_count += 1
            else:
                out_count += 1

        if found_for_sample:
            samples_covered += 1

    print(f"--- Config max_length={ml}, stride={st} ({len(squad_sample)} SQuAD samples) ---")
    print(f"  Samples covered >= 1 window : {samples_covered}/{len(squad_sample)} ({100*samples_covered/len(squad_sample):.1f}%)")
    print(f"  Total windows generated     : {total_windows}")
    print(f"  exact_aligned               : {exact_count}")
    print(f"  mid_token_boundary          : {mid_token_count}")
    print(f"  truncated                   : {truncated_count}")
    print(f"  out_of_window               : {out_count}\n")


--- Config max_length=384, stride=128 (500 SQuAD samples) ---
  Samples covered >= 1 window : 500/500 (100.0%)
  Total windows generated     : 519
  exact_aligned               : 500
  mid_token_boundary          : 2
  truncated                   : 0
  out_of_window               : 17



--- Config max_length=256, stride=64 (500 SQuAD samples) ---
  Samples covered >= 1 window : 500/500 (100.0%)
  Total windows generated     : 654
  exact_aligned               : 516
  mid_token_boundary          : 2
  truncated                   : 2
  out_of_window               : 134

